# Bonus Lab: Modin

**Module:** Week 3 Day 1 (optional)  
**Kernel:** `.venv-probes` (build it first with `Bonus_Lab_Engine_Probes.md`)


> This is an optional bonus walkthrough. You are not expected to master this tool today.
> The goal is awareness: understand what problem this tool tries to solve, when it may help, and what trade-offs it introduces.


In this course, Modin is an awareness topic, not a core tool. The takeaway is that some libraries try to preserve the Pandas API while changing the execution engine underneath.

Modin's pitch is one changed import: `import modin.pandas as pd` and your pandas code runs on a parallel engine such as Dask or Ray underneath. Notice what we did not do: we did not rewrite the workload.

**Why a separate kernel?** Modin has its own dependency stack and execution engine setup. To keep the main Pandas/Polars environment stable, this notebook runs in `.venv-probes`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

REAL_PATH = Path("data/yellow_tripdata_2024-04.parquet")
SYNTH_PATH = Path("data/synthetic_tripdata.parquet")

if REAL_PATH.exists():
    SOURCE_PATH = REAL_PATH
    print("using real TLC data:", SOURCE_PATH)
else:
    if not SYNTH_PATH.exists():
        rng = np.random.default_rng(42)
        n = 1_000_000
        pickup = pd.to_datetime("2024-04-01") + pd.to_timedelta(
            rng.integers(0, 30 * 24 * 3600, n), unit="s"
        )
        duration_s = rng.integers(60, 5400, n)
        synth = pd.DataFrame(
            {
                "tpep_pickup_datetime": pickup,
                "tpep_dropoff_datetime": pickup + pd.to_timedelta(duration_s, unit="s"),
                "payment_type": rng.integers(0, 5, n),
                "fare_amount": (duration_s / 60 * rng.uniform(2.0, 4.0, n)).round(2),
                "trip_distance": (duration_s / 60 * rng.uniform(0.2, 0.6, n)).round(2),
            }
        )
        SYNTH_PATH.parent.mkdir(exist_ok=True)
        synth.to_parquet(SYNTH_PATH, index=False)
    SOURCE_PATH = SYNTH_PATH
    print("real file not found, using synthetic data:", SOURCE_PATH)

In [ ]:
import importlib.util
import os

modin_available = importlib.util.find_spec("modin") is not None

if not modin_available:
    print("Modin is not installed in this kernel.")
    print('Install it into the probe environment: uv pip install --python .venv-probes "modin[dask]"')
else:
    os.environ.setdefault("MODIN_ENGINE", "dask")
    import modin.pandas as mpd

    print("Modin ready, engine:", os.environ["MODIN_ENGINE"])

In [ ]:
%%time
if modin_available:
    mdf = mpd.read_parquet(SOURCE_PATH)
    mdf["trip_duration_min"] = (
        (mdf["tpep_dropoff_datetime"] - mdf["tpep_pickup_datetime"]).dt.total_seconds() / 60
    )
    modin_answer = mdf[mdf["trip_duration_min"] > 25].groupby("payment_type")["fare_amount"].mean()
    print(type(modin_answer))
    print(modin_answer)
else:
    print("Modin timing skipped.")

Two things to watch for:
1. The first Modin operation starts the engine, which takes a few seconds. That startup cost is real and belongs in your findings.
2. On data this size, scheduling overhead can eat the parallel gains. Modin shines on much larger data and many-core machines.

## Your Turn
Change the business question: **for trips longer than 10 minutes, what is the average fare by payment type?**

In [ ]:
# Your code here

## Bonus Tool Takeaways

| Topic | What students should remember | Do they need it today? |
|---|---|---|
| Dask | Pandas-like API, lazy execution, partitions, `.compute()` | No, awareness only |
| Modin | Tries to keep Pandas code while changing the execution engine | No, awareness only |
| FireDucks | Pandas-like acceleration, but environment-sensitive | No, awareness only |
| Parquet | Columnar storage, smaller files, column pruning, compression trade-offs | Yes, important for data engineering |
